
# TP Jour 1 — Projet Olist : HDFS & Couche Bronze

## Fiche de cadrage du projet

- **Titre du sujet :** Analyse Big Data et prédiction des retards de livraison dans le commerce électronique.
- **Dataset Kaggle retenu :** `olistbr/brazilian-ecommerce`.
- **Source principale :** Brazilian E-Commerce Public Dataset by Olist (9 fichiers CSV relationnels).
- **Clés de jointure :** `order_id`, `customer_id`, `product_id`.
- **Problématique métier :** comprendre les facteurs associés aux retards de livraison et prédire le risque qu'une commande soit livrée après la date estimée.
- **Variables temporelles :** dates de commande, livraison réelle et livraison estimée.
- **Variables quantitatives :** prix, frais de transport, nombre d'articles, délai de livraison, retard.
- **Variables catégorielles :** statut de commande, catégorie de produit, ville et État du client.

### Objectifs de la séance
1. Télécharger et explorer le dataset Olist.
2. Explorer le cluster HDFS (1 NameNode + 3 DataNodes), blocs de 16 Mo et réplication configurée.
3. Ingérer les CSV sur HDFS puis créer les tables Bronze au format Delta Lake.
4. Réaliser le crash-test DataNode prévu dans le TP.
5. Vérifier la traçabilité des sources et les tables Delta créées.


## 1. Téléchargement de vos Datasets Kaggle dans `/workspace/data_raw/<source>/`

Renseignez dans la liste `KAGGLE_DATASETS` un ou plusieurs identifiants Kaggle (ex. `["auteur1/dataset-principal", "auteur2/meteo-ou-referentiel"]`), ou déposez directement vos fichiers/sous-dossiers dans `data_raw/`.
- Chaque dataset est isolé dans son propre sous-dossier `data_raw/<nom-dataset>/` pour éviter toute collision de noms (`train.csv`, `metadata.csv`, etc.) entre plusieurs sources.
- **Critère important :** Vérifiez qu'au moins un fichier dépasse **20 à 32 Mo** afin d'observer sa division en plusieurs blocs HDFS de **16 Mo** !

In [1]:
import re
import shutil
from pathlib import Path

DATA_RAW = Path("/workspace/data_raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)
SUPPORTED_EXTS = {".csv", ".parquet", ".json", ".jsonl", ".tsv"}

# Dataset public Olist (9 fichiers CSV relationnels)
KAGGLE_DATASETS = ["olistbr/brazilian-ecommerce"]

target_dir = DATA_RAW / "olist"
target_dir.mkdir(parents=True, exist_ok=True)
if not list(target_dir.glob("*.csv")):
    import kagglehub
    cache_dir = Path(kagglehub.dataset_download(KAGGLE_DATASETS[0]))
    for src_file in sorted(cache_dir.rglob("*")):
        if src_file.is_file() and src_file.suffix.lower() in SUPPORTED_EXTS:
            shutil.copy2(src_file, target_dir / src_file.name)
else:
    print("CSV déjà présents dans data_raw/olist : téléchargement ignoré")

local_files = sorted(
    f for f in DATA_RAW.rglob("*")
    if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS
)
assert local_files, "Aucun CSV trouvé dans /workspace/data_raw/."

print(f"{len(local_files)} fichier(s) détecté(s) dans {DATA_RAW}")
for f in local_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    blocks = max(1, (f.stat().st_size + 16 * 1024 * 1024 - 1) // (16 * 1024 * 1024))
    print(f"{f.relative_to(DATA_RAW)} : {size_mb:.2f} Mo (~{blocks} bloc(s) HDFS)")

CSV déjà présents dans data_raw/olist : téléchargement ignoré
27 fichier(s) détecté(s) dans /workspace/data_raw
brazilian-ecommerce/olist_customers_dataset.csv : 8.62 Mo (~1 bloc(s) HDFS)
brazilian-ecommerce/olist_geolocation_dataset.csv : 58.44 Mo (~4 bloc(s) HDFS)
brazilian-ecommerce/olist_order_items_dataset.csv : 14.72 Mo (~1 bloc(s) HDFS)
brazilian-ecommerce/olist_order_payments_dataset.csv : 5.51 Mo (~1 bloc(s) HDFS)
brazilian-ecommerce/olist_order_reviews_dataset.csv : 13.78 Mo (~1 bloc(s) HDFS)
brazilian-ecommerce/olist_orders_dataset.csv : 16.84 Mo (~2 bloc(s) HDFS)
brazilian-ecommerce/olist_products_dataset.csv : 2.27 Mo (~1 bloc(s) HDFS)
brazilian-ecommerce/olist_sellers_dataset.csv : 0.17 Mo (~1 bloc(s) HDFS)
brazilian-ecommerce/product_category_name_translation.csv : 0.00 Mo (~1 bloc(s) HDFS)
olist/olist_customers_dataset.csv : 8.62 Mo (~1 bloc(s) HDFS)
olist/olist_geolocation_dataset.csv : 58.44 Mo (~4 bloc(s) HDFS)
olist/olist_order_items_dataset.csv : 14.72 Mo (~1 bloc(

## 2. Démarrage de la `SparkSession` pré-configurée sur HDFS

Grâce au fichier `config/spark-defaults.conf`, Spark pointe automatiquement vers `hdfs://namenode:9000` avec **Delta Lake** activé.

### Exercice 2.1 — Initialiser `SparkSession` et contrôler la configuration HDFS
1. Créez une session Spark nommée `"J1_HDFS_Ingestion_Bronze"`.
2. Récupérez via `spark.conf.get(...)` la valeur de `"spark.hadoop.fs.defaultFS"`, `"spark.hadoop.dfs.blocksize"` et `"spark.hadoop.dfs.replication"`.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J1_HDFS_Ingestion_Bronze")
    .getOrCreate()
)

default_fs = spark.sparkContext._jsc.hadoopConfiguration().get("fs.defaultFS")
block_size = spark.sparkContext._jsc.hadoopConfiguration().get("dfs.blocksize")
replication = spark.sparkContext._jsc.hadoopConfiguration().get("dfs.replication")

print("Spark version       :", spark.version)
print("Default FileSystem  :", default_fs)
print("HDFS Block Size (o) :", block_size)
print("HDFS Replication    :", replication)

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-dfb241e0-8589-425b-aa6c-3e03dd71b98a;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found

Spark version       : 4.2.0
Default FileSystem  : hdfs://namenode:9000
HDFS Block Size (o) : 16777216
HDFS Replication    : 2


## 3. Copie des fichiers bruts vers HDFS & Inspection des Blocs de 16 Mo

La cellule suivante téléverse vos fichiers bruts de `/workspace/data_raw` vers `/lakehouse/bronze/raw/` sur HDFS, puis interroge le NameNode pour afficher la localisation exacte de chaque bloc de 16 Mo sur les 3 DataNodes (`datanode1`, `datanode2`, `datanode3`).

In [3]:
jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
PathHDFS = jvm.org.apache.hadoop.fs.Path

raw_hdfs_dir = PathHDFS("/lakehouse/bronze/raw")
fs.mkdirs(raw_hdfs_dir)

for file_path in local_files:
    rel_posix = file_path.relative_to(DATA_RAW).as_posix()
    src = PathHDFS(f"file://{file_path}")
    dst = PathHDFS(f"/lakehouse/bronze/raw/{rel_posix}")
    fs.mkdirs(dst.getParent())
    fs.copyFromLocalFile(False, True, src, dst)
    print(f"Téléversé vers HDFS : {dst}")

Téléversé vers HDFS : /lakehouse/bronze/raw/brazilian-ecommerce/olist_customers_dataset.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/brazilian-ecommerce/olist_geolocation_dataset.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/brazilian-ecommerce/olist_order_items_dataset.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/brazilian-ecommerce/olist_order_payments_dataset.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/brazilian-ecommerce/olist_order_reviews_dataset.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/brazilian-ecommerce/olist_orders_dataset.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/brazilian-ecommerce/olist_products_dataset.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/brazilian-ecommerce/olist_sellers_dataset.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/brazilian-ecommerce/product_category_name_translation.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/olist/olist_customers_dataset.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/olist/olist_geolocation_dataset.c

26/11/21 13:35:02 WARN DataStreamer: Exception in createBlockOutputStream blk_1073742275_1455
java.io.EOFException: Unexpected EOF while trying to read response from server
	at org.apache.hadoop.hdfs.protocolPB.PBHelperClient.vintPrefixed(PBHelperClient.java:532)
	at org.apache.hadoop.hdfs.DataStreamer.createBlockOutputStream(DataStreamer.java:1931)
	at org.apache.hadoop.hdfs.DataStreamer.setupPipelineForCreate(DataStreamer.java:1842)
	at org.apache.hadoop.hdfs.DataStreamer.work(DataStreamer.java:752)
	at org.apache.hadoop.util.Daemon.run(Daemon.java:76)
26/11/21 13:35:02 WARN DataStreamer: Error Recovery for BP-1590342591-172.26.0.2-1795008710306:blk_1073742275_1455 in pipeline [DatanodeInfoWithStorage[172.26.0.5:9866,DS-7c0a420e-e9cb-4e37-8ca0-b11e1a3541b7,DISK], DatanodeInfoWithStorage[172.26.0.3:9866,DS-c5362eb1-74ce-4955-982b-fd30a72c37eb,DISK]]: datanode 0(DatanodeInfoWithStorage[172.26.0.5:9866,DS-7c0a420e-e9cb-4e37-8ca0-b11e1a3541b7,DISK]) is bad.
26/11/21 13:35:02 WARN DataStr

Py4JJavaError: An error occurred while calling o45.copyFromLocalFile.
: org.apache.hadoop.ipc.RemoteException(java.io.IOException): File /lakehouse/bronze/raw/olist/olist_orders_dataset.csv could only be written to 0 of the 1 minReplication nodes. There are 3 datanode(s) running and 3 node(s) are excluded in this operation.
	at org.apache.hadoop.hdfs.server.blockmanagement.BlockManager.chooseTarget4NewBlock(BlockManager.java:2487)
	at org.apache.hadoop.hdfs.server.namenode.FSDirWriteFileOp.chooseTargetForNewBlock(FSDirWriteFileOp.java:294)
	at org.apache.hadoop.hdfs.server.namenode.FSNamesystem.getAdditionalBlock(FSNamesystem.java:3108)
	at org.apache.hadoop.hdfs.server.namenode.NameNodeRpcServer.addBlock(NameNodeRpcServer.java:934)
	at org.apache.hadoop.hdfs.protocolPB.ClientNamenodeProtocolServerSideTranslatorPB.addBlock(ClientNamenodeProtocolServerSideTranslatorPB.java:595)
	at org.apache.hadoop.hdfs.protocol.proto.ClientNamenodeProtocolProtos$ClientNamenodeProtocol$2.callBlockingMe

In [ ]:
# Inspection récursive du découpage en blocs de 16 Mo sur l'ensemble des sources ingérées
remote_iter = fs.listFiles(raw_hdfs_dir, True)
statuses = []
while remote_iter.hasNext():
    statuses.append(remote_iter.next())
statuses.sort(key=lambda s: s.getLen(), reverse=True)
largest_hdfs_path = statuses[0].getPath().toString()

for status in statuses:
    path = status.getPath()
    rel_hdfs = path.toString().split("/lakehouse/bronze/raw/", 1)[-1]
    length_mb = status.getLen() / (1024 * 1024)
    blocks = fs.getFileBlockLocations(status, 0, status.getLen())
    print(f"Fichier : {rel_hdfs} ({length_mb:.2f} Mo | repl={status.getReplication()})")
    for idx, blk in enumerate(blocks):
        print(f"   Bloc #{idx} ({blk.getLength() / (1024*1024):5.2f} Mo) -> DataNodes : {list(blk.getHosts())}")

### Questions d'analyse (à vérifier aussi sur http://localhost:9870)
1. Comparez sur l'interface Web du NameNode (`Utilities > Browse the file system`) le nombre de blocs de votre fichier le plus volumineux avec sa taille divisée par 16 Mo.
2. Un fichier de 3 Mo bloque-t-il réellement 16 Mo d'espace disque sur les DataNodes ? Pourquoi ?
3. Pourquoi avoir configuré `dfs.replication = 2` sur un cluster à `3` DataNodes ?

---
## 4. Crash-Test en Direct : Panne d'un DataNode et Auto-Réparation (~50s)

Dans `config/hdfs-site.xml`, le délai de détection d'un DataNode mort a été abaissé de 10m30s à **~50 secondes** :
$$2 \times \text{recheck-interval (10s)} + 10 \times \text{heartbeat.interval (3s)} = 50\text{ s}$$

### Protocole à réaliser depuis votre terminal :
1. Inspectez l'état initial des blocs : `task hdfs:fsck`
2. Arrêtez brutalement le DataNode 2 :
   ```bash
   task hdfs:crash-dn2   # équivalent à : docker stop datanode2
   ```
3. Observez l'onglet **Datanodes** sur `http://localhost:9870` pendant ~50s : dès que `datanode2` passe à **Dead**, le NameNode détecte les blocs sous-répliqués et ordonne leur copie automatique vers le 3e DataNode survivant !

### Exercice 4.1 — Vérifier la continuité de service et la *Data Locality*
Pendant que `datanode2` est éteint, lisez votre plus gros fichier (`largest_hdfs_path`) avec PySpark, affichez `df_check.count()` ainsi que `df_check.rdd.getNumPartitions()`, et expliquez le lien avec le nombre de blocs HDFS.

*(Pensez ensuite à rallumer le nœud avec `task hdfs:restore-dn2` !)*

In [ ]:
print("Fichier cible du test de résilience :", largest_hdfs_path)

lower_path = largest_hdfs_path.lower()
if lower_path.endswith(".csv") or lower_path.endswith(".tsv"):
    df_check = (
        spark.read.option("header", True)
        .option("inferSchema", True)
        .option("encoding", "UTF-8")
        .csv(largest_hdfs_path)
    )
elif lower_path.endswith(".parquet"):
    df_check = spark.read.parquet(largest_hdfs_path)
elif lower_path.endswith(".json") or lower_path.endswith(".jsonl"):
    df_check = spark.read.json(largest_hdfs_path)
else:
    raise ValueError(f"Format non pris en charge : {largest_hdfs_path}")

print(f"Nombre total de lignes lues       : {df_check.count():,}")
print(f"Nombre de partitions Spark créées : {df_check.rdd.getNumPartitions()}")

---
## 5. Construction de votre Couche Bronze en Delta Lake (`/lakehouse/bronze/`)

### Exercice 5.1 — Convertir vos fichiers bruts (mono ou multi-datasets) en tables Delta Lake Bronze
Dans une architecture **Médaillon multi-sources**, la couche **Bronze** conserve chaque table brute fidèle à sa source tout en l'enrichissant de métadonnées de lignage (*lineage*) indispensables avant les jointures du Jour 2 :
- `_ingested_at` : `F.current_timestamp()`
- `_source_dataset` : `F.lit(dataset_name)` *(identifie le dataset d'origine)*
- `_source_file` : `F.col("_metadata.file_path")`

> **Attention aux noms de colonnes Delta/Parquet :** Si vos fichiers CSV bruts contiennent des espaces ou caractères spéciaux dans les en-têtes, normalisez-les simplement en `snake_case` (ex. `.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])`).

Complétez la cellule ci-dessous pour ingérer chacune des tables utiles de vos datasets dans `/lakehouse/bronze/<nom_table>`.

In [ ]:
# Chaque fichier CSV Olist est conservé comme table Bronze indépendante.
OLIST_FILES = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

BRONZE_SOURCES = {
    table: ("olist", f"/lakehouse/bronze/raw/olist/{filename}")
    for table, filename in OLIST_FILES.items()
}

for table_name, (dataset_name, raw_path) in BRONZE_SOURCES.items():
    df_raw = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("encoding", "UTF-8")
        .csv(raw_path)
    )

    # Normalisation des noms de colonnes
    normalized = [
        re.sub(r"[^a-zA-Z0-9_]", "_", c.strip().lower())
        for c in df_raw.columns
    ]
    df_raw = df_raw.toDF(*normalized)

    df_bronze = (
        df_raw
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_dataset", F.lit(dataset_name))
        .withColumn("_source_file", F.lit(raw_path))
    )

    bronze_delta_path = f"/lakehouse/bronze/{table_name}"
    (
        df_bronze.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(bronze_delta_path)
    )

    cnt = spark.read.format("delta").load(bronze_delta_path).count()
    print(f"[BRONZE OK] {bronze_delta_path} -> {cnt:,} lignes")

In [ ]:
# Auto-contrôle de fin de Jour 1 : au moins une table Delta valide dans /lakehouse/bronze/
bronze_root = PathHDFS("/lakehouse/bronze")
bronze_delta_tables = [
    s.getPath().getName()
    for s in fs.listStatus(bronze_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(bronze_delta_tables) > 0, "Aucune table Delta détectée dans /lakehouse/bronze/ !"
print("Tables Delta Bronze validées :", bronze_delta_tables)